il modello della lezione usa una strategia Direct Vector Output (prevede tutto in un solo colpo)

- modifica: cambia il layer LSTM in una GRU (Gated Recurrent Unit) e riduci il numero di neuroni a 64

Sperimentazione: porta l'orizzonte di previsione N_OUT a 12 mesi

Analisi: osserva il grafico dei residue: l'errore al 12 mese è significativamente più alto rispetto al 1  mese? spiega perchè la rete fatica man mano che si allontana nel tempo (concetto di entropia informativa)

In [ ]:
import os

# =================================================================
# 1. CONFIGURAZIONE AMBIENTE (Keras 3 Multi-Backend)
# =================================================================

# Impostiamo il backend su 'torch'. In Keras 3, questa variabile deve essere 
# definita PRIMA di importare keras per istruire il framework a usare 
# PyTorch come motore per il calcolo dei gradienti e dei tensori.
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# Fissiamo i semi (seeds) per rendere l'esperimento riproducibile.
# Questo influenza l'inizializzazione dei pesi della rete e la generazione del rumore.
keras.utils.set_random_seed(42)

# =================================================================
# 2. GENERAZIONE DATASET SINTETICO
# =================================================================

# Creiamo un indice temporale mensile per 13 anni (156 mesi).
time_index = pd.date_range(start='2010-01-01', periods=156, freq='MS')

# Definiamo le componenti della serie storica: y(t) = Trend + Seasonality + Noise
# Trend: Una crescita lineare nel tempo (da 500 a 1000 pounds).
trend = np.linspace(500, 1000, 156) 

# Stagionalità: Una funzione seno per simulare il ciclo annuale naturale.
# 2*pi*n / 12 assicura che il ciclo si completi esattamente ogni 12 mesi.
seasonality = 100 * np.sin(2 * np.pi * np.arange(156) / 12) 

# Rumore: Fluttuazioni casuali (distribuzione normale) per simulare l'incertezza reale.
noise = np.random.normal(0, 10, 156) 

# Assembliamo il DataFrame finale di Pandas.
df = pd.DataFrame({'Production': trend + seasonality + noise}, index=time_index)
df.index.name = 'Month'

# =================================================================
# 3. ENCODING CICLICO DEL TEMPO
# =================================================================

# Problema: Se usiamo i mesi (1-12), il modello vede Dicembre (12) e Gennaio (1) 
# come numericamente distanti, mentre sono temporalmente vicini.
# Soluzione: Mappiamo il tempo su un cerchio unitario usando seno e coseno.
month_series = df.index.month
df['sin_month'] = np.sin(2 * np.pi * month_series / 12)
df['cos_month'] = np.cos(2 * np.pi * month_series / 12)

# 

# =================================================================
# 4. NORMALIZZAZIONE (Scaling)
# =================================================================

# Le LSTM sono sensibili alla scala dei dati. Usiamo MinMaxScaler per portare 
# tutto nell'intervallo [0, 1]. Questo aiuta la funzione di attivazione tanh 
# all'interno della LSTM a lavorare nella sua zona di massima pendenza (gradiente).
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Production', 'sin_month', 'cos_month']].values)

# =================================================================
# 5. FINESTRATURA (Windowing) PER PREDIZIONE MULTI-STEP
# =================================================================

def create_multistep_window(data, n_input, n_out):
    """
    Trasforma la serie piatta in un problema di apprendimento supervisionato.
    n_input: Quanti mesi passati guardiamo (Look-back window).
    n_out: Quanti mesi futuri vogliamo prevedere (Forecast horizon).
    """
    X, y = [], []
    for i in range(len(data)):
        end_ix = i + n_input
        out_end_ix = end_ix + n_out
        # Ci fermiamo quando non abbiamo abbastanza dati per completare l'orizzonte futuro.
        if out_end_ix > len(data): break
        
        # X: Matrice con (Mesi, Features). Features sono Production, Sin e Cos.
        X.append(data[i:end_ix, :])
        # y: Vettore target con solo la Produzione per i passi successivi.
        y.append(data[end_ix:out_end_ix, 0])
    return np.array(X), np.array(y)

# 

# Guardiamo 12 mesi per prevederne 12 contemporaneamente.
N_INPUT, N_OUT = 12, 12
X, y = create_multistep_window(scaled_data, N_INPUT, N_OUT)

# Dividiamo i dati: 80% per l'addestramento, 20% per il test finale (split cronologico).
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# =================================================================
# 6. ARCHITETTURA DEL MODELLO LSTM
# =================================================================

model = keras.Sequential([
    # Input: Riceve sequenze di 12 passi temporali con 3 variabili ciascuno.
    keras.layers.Input(shape=(N_INPUT, 3)),
    
    # GRU Layer: 64 unità. La tangente iperbolica ('tanh') è l'attivazione standard.
    # return_sequences=False restituisce solo l'ultimo stato utile a predire il futuro.
    keras.layers.GRU(64, activation='tanh', return_sequences=False),
    
    # Dropout: Spegne casualmente il 20% delle connessioni per evitare l'overfitting.
    keras.layers.Dropout(0.2),
    
    # Dense: Uno strato finale con N_OUT neuroni (6). Ogni neurone predice un mese futuro.
    keras.layers.Dense(N_OUT)
])

# AdamW: Variante moderna dell'ottimizzatore Adam con gestione migliorata del Weight Decay.
model.compile(optimizer='adamw', loss='mse', metrics=['mae'])

# =================================================================
# 7. ADDESTRAMENTO
# =================================================================

print("[INFO] Addestramento in corso con backend PyTorch...")
# epochs=150: Il modello vedrà i dati 150 volte. 
# batch_size=16: I pesi vengono aggiornati ogni 16 esempi.
model.fit(X_train, y_train, epochs=150, batch_size=16, 
          validation_data=(X_test, y_test), verbose=0)

# =================================================================
# 8. VALUTAZIONE E INVERSIONE DELLO SCALING
# =================================================================

# Generiamo le predizioni sui dati di test.
predictions = model.predict(X_test, verbose=0)

# Dobbiamo riportare i dati dalla scala [0,1] all'unità originale (Pounds).
# Creiamo uno scaler "fittizio" che conosce solo i parametri della colonna Production.
production_scaler = MinMaxScaler()
production_scaler.min_, production_scaler.scale_ = scaler.min_[0], scaler.scale_[0]

y_test_real = production_scaler.inverse_transform(y_test)
y_preds_real = production_scaler.inverse_transform(predictions)

# Residui: Differenza tra il valore reale e quello predetto (Errore).
residuals = y_test_real - y_preds_real

# =================================================================
# 9. VISUALIZZAZIONE RISULTATI (Plotting)
# =================================================================

plt.figure(figsize=(18, 10))

# SUBPLOT 1: Confronto Reale vs Predetto sul primo mese dell'orizzonte (M+1).
plt.subplot(2, 1, 1)
plt.plot(df.index[-len(y_test_real):], y_test_real[:, 0], label='Dati Reali', color='black', linewidth=2)
plt.plot(df.index[-len(y_preds_real):], y_preds_real[:, 0], label='Predizione M+1', color='red', linestyle='--')
plt.title("Analisi Time Series: Reale vs Predetto")
plt.ylabel("Pounds")
plt.legend()
plt.grid(True, alpha=0.3)

# SUBPLOT 2: Degradazione dell'errore. 
# Più ci allontaniamo nel tempo (da M+1 a M+6), più l'errore tende a salire.
plt.subplot(2, 2, 3)
mae_per_step = np.mean(np.abs(residuals), axis=0)
plt.bar([f"M+{i+1}" for i in range(N_OUT)], mae_per_step, color='teal')
plt.title("Errore Medio (MAE) per Step Futuro")
plt.ylabel("Pounds")

# 

# SUBPLOT 3: Distribuzione dei Residui. 
# Se l'errore segue una campana di Gauss centrata sullo zero, il modello non ha bias sistematici.
plt.subplot(2, 2, 4)
plt.hist(residuals.flatten(), bins=20, color='orange', edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--')
plt.title("Distribuzione dell'Errore Residuo")
plt.xlabel("Residuo (Pounds)")

plt.tight_layout()
plt.show()

print(f"[LOG] Errore medio al primo mese (M+1): {mae_per_step[0]:.2f} Pounds")